# 2 Graph 的基本要素

一个 Graph 由 4 个要素构成：

| 要素 | 含义 | 代码对应 |
|---|---|---|
| **State** | 所有 Node 共享的数据容器 | 一个 TypedDict 或 Pydantic 类 |
| **Node** | 一个工作单元：普通 Python 函数 | `graph.add_node(...)` |
| **Edge** | Node 间的连接路径，决定执行顺序 | `graph.add_edge(...)` / `add_conditional_edges(...)` |
| **START / END** | 特殊 Node：Graph 的入口和出口 | `START`、`END` 常量 |

## 2.1 State

State 是一个类型定义，描述 Graph 上流动的数据。每个 Node 读 State、返回**部分更新**，LangGraph 自动把更新合并回 State：

```python
from typing_extensions import TypedDict

class ChatState(TypedDict):
    messages: list[str]   # 对话历史
    count: int            # 处理轮数
```

## 2.2 Node

Node 就是一个函数：**接收 State，返回要更新的字段**：

```python
def node_a(state: ChatState) -> dict:
    return {"count": state["count"] + 1}   # 只返回要更新的字段
```

## 2.3 Edge

- **普通 Edge**：`A → B`，A 跑完一定跑 B
- **条件 Edge**：A 跑完后根据返回值**选择**下一个 Node

## 2.4 拼装 StateGraph

```python
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)          # 1. 创建 Graph，声明 State 类型
builder.add_node("node_a", node_a)      # 2. 添加 Node
builder.add_edge(START, "node_a")       # 3. 连接：入口 → Node
builder.add_edge("node_a", END)         # 4. 连接：Node → 出口
app = builder.compile()                  # 5. 编译
```

下面用完整代码演示一个「两个 Node + State 累积」的最小 Graph：

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class OverAllState(TypedDict):
    logs: list[str]
    cur_id: str


def node_1(state: OverAllState) -> dict:
    # 无 reducer 的 key 是「后写覆盖先写」，想保留旧值就自己读出来拼上
    return {
        "cur_id": state["cur_id"] + ", node_1",
        "logs": state["logs"] + ["node_1 运行完毕"],
    }


def node_2(state: OverAllState) -> dict:
    return {
        "cur_id": state["cur_id"] + ", node_2",
        "logs": state["logs"] + ["node_2 运行完毕"],
    }


builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

result = builder.compile().invoke({"cur_id": "start", "logs": []})
print(result)
# {'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}

# 「自己读旧值拼新值」很啰嗦。第 2 章的 reducer（Annotated[T, add]）可以让 Node 只返回增量，
# 合并交给 LangGraph 自动完成。